# Model 2: Real Estate Investment ROI Analysis

Objective: Compare cash purchases and mortgage-financed purchases under a $10 million investment budget.

Metrics:
- Net Operating Income (NOI)
- Annual Cash Flow
- Cash-on-Cash Return
- Return on Investment (ROI)

# Import Libraries

In [66]:
from pathlib import Path
import pandas as pd
import numpy as np


# Set global variables

In [67]:
TOTAL_BUDGET = 10_000_000

## Assumptions

In [68]:
DOWN_PAYMENT_RATE = 0.25
ANNUAL_INTEREST_RATE = 0.05
AMORTIZATION_YEARS = 25

# Mortgage Calculator

In [69]:
def monthly_mortgage_payment(property_price, down_payment_rate=0.25, annual_interest_rate=0.05, amortization_years=25):
    
    loan_amount = property_price * (1 - down_payment_rate)

    monthly_rate = annual_interest_rate / 12

    number_of_payments = amortization_years * 12

    if loan_amount == 0:
        return 0.0

    if monthly_rate == 0:
        return loan_amount / number_of_payments

    payment = loan_amount * monthly_rate * (1 + monthly_rate) ** number_of_payments/ ((1 + monthly_rate) ** number_of_payments - 1)

    return payment

# Investment Return Calculator

In [70]:
def calculate_investment(property_price, annual_revenue, annual_operating_expenses, down_payment_rate=0.25, annual_interest_rate=0.05, amortization_years=25):
    
    down_payment = property_price * down_payment_rate
    
    loan_amount = property_price - down_payment
    
    # how much we need to pay per month
    monthly_payment = monthly_mortgage_payment(property_price, down_payment_rate, annual_interest_rate, amortization_years)

    # how much we need to pay per year
    annual_mortgage_payment = monthly_payment * 12

    # how much money we will earn
    net_operating_income = annual_revenue - annual_operating_expenses
    annual_cash_flow = net_operating_income - annual_mortgage_payment

    # OMG ITS COC!!!!!
    cash_on_cash_return = (
        annual_cash_flow / down_payment
        if down_payment > 0
        else np.nan
    )

    
    cap_rate = (
        net_operating_income / property_price
        if property_price > 0
        else np.nan
    )

    return {
        "property_price": property_price,
        "annual_revenue": annual_revenue,
        "annual_operating_expenses": annual_operating_expenses,
        "net_operating_income": net_operating_income,
        "loan_amount": loan_amount,
        "down_payment": down_payment,
        "annual_mortgage_payment": annual_mortgage_payment,
        "annual_cash_flow": annual_cash_flow,
        "cash_on_cash_return": cash_on_cash_return,
        "cap_rate": cap_rate
    }


## Example

In [71]:
example = calculate_investment(
    property_price=1_000_000,
    annual_revenue=60_000,
    annual_operating_expenses=20_000,
    down_payment_rate=0.25
)

example_df = pd.DataFrame([example])

print(example_df.T.round(2))

                                    0
property_price             1000000.00
annual_revenue               60000.00
annual_operating_expenses    20000.00
net_operating_income         40000.00
loan_amount                 750000.00
down_payment                250000.00
annual_mortgage_payment      52613.10
annual_cash_flow            -12613.10
cash_on_cash_return             -0.05
cap_rate                         0.04


# Compare Cash Purchase vs Mortgage

In [72]:
cash_purchase = calculate_investment(
    property_price=1_000_000,
    annual_revenue=60_000,
    annual_operating_expenses=20_000,
    down_payment_rate=1.0
)

mortgage_purchase = calculate_investment(
    property_price=1_000_000,
    annual_revenue=60_000,
    annual_operating_expenses=20_000,
    down_payment_rate=0.25
)

financing_comparison = pd.DataFrame([
    {"financing": "Cash", **cash_purchase},
    {"financing": "Mortgage (25% Down)", **mortgage_purchase}
])

print(
    financing_comparison[
        [
            "financing",
            "down_payment",
            "loan_amount",
            "net_operating_income",
            "annual_mortgage_payment",
            "annual_cash_flow",
            "cash_on_cash_return",
            "cap_rate"
        ]
    ].round(3).to_string(index=False)
)


          financing  down_payment  loan_amount  net_operating_income  annual_mortgage_payment  annual_cash_flow  cash_on_cash_return  cap_rate
               Cash     1000000.0          0.0                 40000                    0.000         40000.000                 0.04      0.04
Mortgage (25% Down)      250000.0     750000.0                 40000                52613.104        -12613.104                -0.05      0.04


## Neighbourhood and Property Type Analysis

In [73]:
# Load the provided dataset without relying on cleaning.py.
# Search common project paths so the notebook works across folder layouts.
candidate_paths = [
    Path("../../data/output/final_df.csv"),
    Path("../data/output/final_df.csv"),
    Path("data/output/final_df.csv"),
    Path("final_df.csv"),
    Path("/mnt/data/final_df.csv"),  # ChatGPT sandbox copy
]

csv_path = next((p for p in candidate_paths if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Cannot find final_df.csv. Put it beside this notebook, "
        "or update candidate_paths with its location."
    )

airbnb = pd.read_csv(csv_path)
print(f"Loaded {len(airbnb):,} rows from {csv_path}")

required_columns = [
    "neighbourhood", "property_category", "price",
    "estimated_occupancy_l365d", "estimated_revenue_l365d"
]
missing = [column for column in required_columns if column not in airbnb.columns]
if missing:
    raise KeyError(f"Missing required columns in final_df.csv: {missing}")

for column in ["price", "estimated_occupancy_l365d", "estimated_revenue_l365d"]:
    airbnb[column] = pd.to_numeric(airbnb[column], errors="coerce")

# Historical estimated revenue is a descriptive market benchmark,
# not confirmed booking income or a validated investment forecast.
market = airbnb.loc[
    airbnb["estimated_revenue_l365d"].notna()
    & airbnb["price"].gt(0)
    & airbnb["neighbourhood"].notna()
    & airbnb["property_category"].notna()
].copy()

# .size counts rows without requiring a nonexistent 'listing_id' column.
property_market = (
    market.groupby(["neighbourhood", "property_category"])
    .agg(
        listings=("id", "size"),
        median_nightly_price=("price", "median"),
        median_occupied_nights=("estimated_occupancy_l365d", "median"),
        median_annual_revenue=("estimated_revenue_l365d", "median"),
        mean_annual_revenue=("estimated_revenue_l365d", "mean")
    )
    .reset_index()
)

property_market["sample_size_flag"] = np.where(
    property_market["listings"] < 10, "Small sample", "OK"
)
property_market = property_market.sort_values(
    "median_annual_revenue", ascending=False
)

print("\nNEIGHBOURHOOD + PROPERTY TYPE COMPARISON")
print(property_market.round(2).to_string(index=False))

output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
export_path = output_dir / "neighbourhood_property_market.csv"
property_market.to_csv(export_path, index=False)
print(f"\nSaved: {export_path} ({len(property_market)} groups)")


Loaded 6,509 rows from ..\..\data\output\final_df.csv

NEIGHBOURHOOD + PROPERTY TYPE COMPARISON
           neighbourhood              property_category  listings  median_nightly_price  median_occupied_nights  median_annual_revenue  mean_annual_revenue sample_size_flag
                Oakridge                          Other         1               1146.88                   255.0               292454.0            292454.00     Small sample
           Arbutus Ridge              Entire Townhouses         3                752.04                   240.0               180490.0            145622.00     Small sample
                Fairview              Entire Townhouses         4                766.50                    75.0               104460.0           1421645.00     Small sample
                West End       Commercial / Hospitality         3                356.00                   246.0                87576.0             79892.00     Small sample
               Killarney               

## Filter only the ones with at least 20 listings

In [74]:
roi_groups = property_market[
    property_market["listings"] >= 20
].copy()

roi_groups = roi_groups.sort_values(
    ["neighbourhood", "property_category"]
)

print(roi_groups)


roi_groups.to_csv(
    "output/roi_groups.csv",
    index=False
)

                neighbourhood               property_category  listings  \
1               Arbutus Ridge                   Entire Houses        52   
4               Arbutus Ridge                   Private Rooms        45   
5               Arbutus Ridge  Secondary Suites / Guest Units        27   
7                    Downtown      Entire Apartments / Condos      1275   
11                   Downtown                   Private Rooms        74   
13          Downtown Eastside      Entire Apartments / Condos       308   
14          Downtown Eastside                   Entire Houses        45   
17          Downtown Eastside                   Private Rooms        21   
20          Dunbar Southlands                   Entire Houses       114   
22          Dunbar Southlands                   Private Rooms       109   
23          Dunbar Southlands  Secondary Suites / Guest Units        48   
24                   Fairview      Entire Apartments / Condos        90   
30         Grandview-Wood

## Property Assessment Data

In [75]:
property_df = pd.read_csv("../../data/input/property-tax-report.csv", sep=';')

print("DATASET SHAPE:")
print(property_df.shape)

print("\nAVAILABLE COLUMNS:")
print(property_df.columns.tolist())

print("\nFIRST 5 ROWS:")
print(property_df.head().to_string(index=False))

# Check reporting year
if "REPORT_YEAR" in property_df.columns:
    print("\nREPORT YEARS:")
    print(property_df["REPORT_YEAR"].value_counts())

# Check the available property classifications
for col in ["LEGAL_TYPE", "CURRENT_LAND_VALUE",
            "CURRENT_IMPROVEMENT_VALUE"]:
    if col in property_df.columns:
        print(f"\n{col}:")
        print(property_df[col].head(10))

DATASET SHAPE:
(229098, 30)

AVAILABLE COLUMNS:
['PID', 'LEGAL_TYPE', 'FOLIO', 'LAND_COORDINATE', 'ZONING_DISTRICT', 'ZONING_CLASSIFICATION', 'LOT', 'PLAN', 'BLOCK', 'DISTRICT_LOT', 'FROM_CIVIC_NUMBER', 'TO_CIVIC_NUMBER', 'STREET_NAME', 'PROPERTY_POSTAL_CODE', 'NARRATIVE_LEGAL_LINE1', 'NARRATIVE_LEGAL_LINE2', 'NARRATIVE_LEGAL_LINE3', 'NARRATIVE_LEGAL_LINE4', 'NARRATIVE_LEGAL_LINE5', 'CURRENT_LAND_VALUE', 'CURRENT_IMPROVEMENT_VALUE', 'TAX_ASSESSMENT_YEAR', 'PREVIOUS_LAND_VALUE', 'PREVIOUS_IMPROVEMENT_VALUE', 'YEAR_BUILT', 'BIG_IMPROVEMENT_YEAR', 'TAX_LEVY', 'NEIGHBOURHOOD_CODE', 'REPORT_YEAR', 'NOTE']

FIRST 5 ROWS:
        PID LEGAL_TYPE        FOLIO  LAND_COORDINATE ZONING_DISTRICT     ZONING_CLASSIFICATION LOT     PLAN BLOCK DISTRICT_LOT FROM_CIVIC_NUMBER  TO_CIVIC_NUMBER       STREET_NAME PROPERTY_POSTAL_CODE               NARRATIVE_LEGAL_LINE1               NARRATIVE_LEGAL_LINE2               NARRATIVE_LEGAL_LINE3               NARRATIVE_LEGAL_LINE4               NARRATIVE_LEGAL_LI


# Clean 2026 Property Assessments

In [76]:
# Standardize numeric fields
value_columns = [
    "CURRENT_LAND_VALUE",
    "CURRENT_IMPROVEMENT_VALUE"
]

for col in value_columns:
    property_df[col] = pd.to_numeric(
        property_df[col],
        errors="coerce"
    )

# Total assessed value
property_df["assessed_value"] = (
    property_df["CURRENT_LAND_VALUE"]
    + property_df["CURRENT_IMPROVEMENT_VALUE"]
)

# Keep usable assessments
property_clean = property_df.loc[
    property_df["REPORT_YEAR"].eq(2026)
    & property_df["assessed_value"].gt(0)
    & property_df["LEGAL_TYPE"].notna()
    & property_df["NEIGHBOURHOOD_CODE"].notna()
].copy()

# Standardize neighbourhood codes
property_clean["NEIGHBOURHOOD_CODE"] = (
    property_clean["NEIGHBOURHOOD_CODE"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(3)
)

# Group by neighbourhood code and legal type
assessment_summary = (
    property_clean
    .groupby(
        ["NEIGHBOURHOOD_CODE", "LEGAL_TYPE"]
    )
    .agg(
        properties=("assessed_value", "count"),
        median_assessed_value=(
            "assessed_value", "median"
        ),
        mean_assessed_value=(
            "assessed_value", "mean"
        )
    )
    .reset_index()
)

print("\nASSESSMENT SUMMARY")
print(assessment_summary.round(2).to_string(index=False))

print("\nNEIGHBOURHOOD CODES")
print(
    sorted(
        property_clean["NEIGHBOURHOOD_CODE"].unique()
    )
)

print("\nLEGAL TYPE COUNTS")
print(property_clean["LEGAL_TYPE"].value_counts())

Path("output").mkdir(exist_ok=True)

assessment_summary.to_csv(
    "output/property_assessment_summary.csv",
    index=False
)



ASSESSMENT SUMMARY
NEIGHBOURHOOD_CODE LEGAL_TYPE  properties  median_assessed_value  mean_assessed_value
               001       LAND        3444              3388500.0           4933652.52
               001      OTHER           6             30564000.0          52802383.33
               001     STRATA         923               996000.0           1215538.36
               002       LAND        4404              2845000.0           4170123.64
               002      OTHER          16               136500.0          25938287.50
               002     STRATA       10638               930373.5           1156762.36
               003       LAND        4378              3171500.0           3631192.91
               003     STRATA         647               998200.0           1251561.17
               004       LAND        2927              3362000.0           4180685.55
               004      OTHER           2              4078400.0           4078400.00
               004     STRATA     

# Map BC Assessment Neighbourhood Codes

In [77]:
assessment_neighbourhoods = {
    "001": "Point Grey",
    "002": "Kitsilano",
    "003": "Dunbar",
    "004": "Arbutus / Mackenzie Heights",
    "005": "Kerrisdale",
    "006": "Southlands",
    "007": "Fairview",
    "008": "Shaughnessy",
    "009": "Cambie",
    "010": "South Granville",
    "011": "Oakridge",
    "012": "Marpole",
    "013": "Mount Pleasant",
    "014": "Grandview",
    "015": "Cedar Cottage",
    "016": "Main / Fraser",
    "017": "South Vancouver",
    "018": "Marine Drive",
    "019": "Knight",
    "020": "Hastings East",
    "021": "Renfrew",
    "022": "Renfrew Heights",
    "023": "Collingwood",
    "024": "Killarney",
    "025": "Fraserview",
    "026": "Downtown",
    "027": "West End",
    "028": "Harbour",
    "029": "Downtown South",
    "030": "False Creek North"
}

property_clean["assessment_neighbourhood"] = (
    property_clean["NEIGHBOURHOOD_CODE"]
    .map(assessment_neighbourhoods)
)

print("UNMAPPED ASSESSMENT CODES:")
print(
    property_clean.loc[
        property_clean["assessment_neighbourhood"].isna(),
        "NEIGHBOURHOOD_CODE"
    ].value_counts()
)

# Calculate values grouped by official assessment area
assessment_by_area = (
    property_clean
    .groupby(
        ["assessment_neighbourhood", "LEGAL_TYPE"]
    )
    .agg(
        properties=("assessed_value", "count"),
        median_assessed_value=("assessed_value", "median")
    )
    .reset_index()
)

print("\nASSESSMENT VALUES BY AREA")
print(
    assessment_by_area
    .sort_values(
        ["assessment_neighbourhood", "LEGAL_TYPE"]
    )
    .to_string(index=False)
)

assessment_by_area.to_csv(
    "output/property_assessment_by_area.csv",
    index=False
)


UNMAPPED ASSESSMENT CODES:
Series([], Name: count, dtype: int64)

ASSESSMENT VALUES BY AREA
   assessment_neighbourhood LEGAL_TYPE  properties  median_assessed_value
Arbutus / Mackenzie Heights       LAND        2927              3362000.0
Arbutus / Mackenzie Heights      OTHER           2              4078400.0
Arbutus / Mackenzie Heights     STRATA        1110              1170500.0
                     Cambie       LAND        2905              3029000.0
                     Cambie      OTHER           3               696881.0
                     Cambie     STRATA        3721              1040000.0
              Cedar Cottage       LAND        3108              1964000.0
              Cedar Cottage     STRATA        2715               848000.0
                Collingwood       LAND        5372              1764500.0
                Collingwood      OTHER           1            154740000.0
                Collingwood     STRATA        7650               590000.0
                   D

# Preliminary Property Acquisition Values

In [78]:
candidates = roi_groups.copy()

# Inspect actual categories
print("AVAILABLE AIRBNB PROPERTY CATEGORIES:")
print(candidates["property_category"].unique())

# Assign preliminary acquisition proxies
def classify_acquisition(category):
    category = str(category).lower()

    if "apartment" in category or "condo" in category:
        return "STRATA"

    if "townhouse" in category:
        return "STRATA"

    if "entire house" in category:
        return "LAND"

    # Private rooms, suites and other categories need
    # a different acquisition model.
    return None

candidates["assessment_legal_type"] = (
    candidates["property_category"]
    .apply(classify_acquisition)
)

# Keep independently purchasable categories
candidates = candidates[
    candidates["assessment_legal_type"].notna()
].copy()

# Prepare assessment data
prices = assessment_by_area.rename(columns={
    "assessment_neighbourhood": "neighbourhood",
    "LEGAL_TYPE": "assessment_legal_type",
    "median_assessed_value": "assessed_price_proxy",
    "properties": "assessment_property_count"
})

# Join on BOTH neighbourhood and assessment category
roi_screening = candidates.merge(
    prices[
        [
            "neighbourhood",
            "assessment_legal_type",
            "assessed_price_proxy",
            "assessment_property_count"
        ]
    ],
    on=["neighbourhood", "assessment_legal_type"],
    how="left",
    validate="many_to_one"
)

# Calculate gross yield for successfully matched groups
roi_screening["gross_yield_proxy"] = (
    roi_screening["median_annual_revenue"]
    / roi_screening["assessed_price_proxy"]
)

# Flag whether a geographic match was found
roi_screening["price_match_status"] = np.where(
    roi_screening["assessed_price_proxy"].notna(),
    "Matched",
    "Needs geography mapping"
)

print("\nMATCH STATUS")
print(roi_screening["price_match_status"].value_counts())

print("\nPRELIMINARY GROSS YIELD COMPARISON")
print(
    roi_screening.loc[
        roi_screening["assessed_price_proxy"].notna(),
        [
            "neighbourhood",
            "property_category",
            "listings",
            "median_annual_revenue",
            "assessed_price_proxy",
            "gross_yield_proxy"
        ]
    ]
    .sort_values("gross_yield_proxy", ascending=False)
    .round(4)
    .to_string(index=False)
)

print("\nUNMATCHED GROUPS")
print(
    roi_screening.loc[
        roi_screening["assessed_price_proxy"].isna(),
        ["neighbourhood", "property_category"]
    ].to_string(index=False)
)

Path("output").mkdir(exist_ok=True)

roi_screening.to_csv(
    "output/roi_screening.csv",
    index=False
)


AVAILABLE AIRBNB PROPERTY CATEGORIES:
['Entire Houses' 'Private Rooms' 'Secondary Suites / Guest Units'
 'Entire Apartments / Condos']

MATCH STATUS
price_match_status
Needs geography mapping    15
Matched                    12
Name: count, dtype: int64

PRELIMINARY GROSS YIELD COMPARISON
 neighbourhood          property_category  listings  median_annual_revenue  assessed_price_proxy  gross_yield_proxy
      Downtown Entire Apartments / Condos      1275                19128.0              643000.0             0.0297
     Killarney              Entire Houses        52                23237.5             1981500.0             0.0117
     Kitsilano              Entire Houses       151                31483.0             2845000.0             0.0111
Mount Pleasant Entire Apartments / Condos       172                 6865.5              727000.0             0.0094
     Kitsilano Entire Apartments / Condos       107                 7191.0              930373.5             0.0077
       Marpole

# Preliminary Cash vs Mortgage comparison

In [79]:
OPERATING_EXPENSE_RATE = 0.30
MORTGAGE_DOWN_PAYMENT = 0.25
MORTGAGE_INTEREST_RATE = 0.05
AMORTIZATION_YEARS = 25

results = []

for _, row in roi_screening.iterrows():

    result = {
        "neighbourhood": row["neighbourhood"],
        "property_category": row["property_category"],
        "listings": row["listings"],
        "price_match_status": row["price_match_status"],
        "annual_revenue": row["median_annual_revenue"],
        "purchase_price_proxy": row["assessed_price_proxy"]
    }

    price = row["assessed_price_proxy"]
    revenue = row["median_annual_revenue"]

    if pd.notna(price) and price > 0:

        expenses = revenue * OPERATING_EXPENSE_RATE

        cash = calculate_investment(
            property_price=price,
            annual_revenue=revenue,
            annual_operating_expenses=expenses,
            down_payment_rate=1.0
        )

        mortgage = calculate_investment(
            property_price=price,
            annual_revenue=revenue,
            annual_operating_expenses=expenses,
            down_payment_rate=MORTGAGE_DOWN_PAYMENT,
            annual_interest_rate=MORTGAGE_INTEREST_RATE,
            amortization_years=AMORTIZATION_YEARS
        )

        result.update({
            "annual_operating_expenses": expenses,
            "NOI": cash["net_operating_income"],
            "cap_rate": cash["cap_rate"],
            "cash_annual_cash_flow": cash["annual_cash_flow"],
            "mortgage_annual_cash_flow": mortgage["annual_cash_flow"],
            "mortgage_cash_on_cash": mortgage["cash_on_cash_return"]
        })

    results.append(result)

roi_comparison = pd.DataFrame(results)

print("\nPRELIMINARY CASH VS MORTGAGE COMPARISON")
print(
    roi_comparison[
        [
            "neighbourhood",
            "property_category",
            "cap_rate",
            "cash_annual_cash_flow",
            "mortgage_annual_cash_flow",
            "mortgage_cash_on_cash"
        ]
    ]
    .sort_values("cap_rate", ascending=False)
    .round(4)
    .to_string(index=False)
)

print("\nMATCH COVERAGE")
print(roi_comparison["price_match_status"].value_counts())

roi_comparison.to_csv(
    "output/preliminary_roi_comparison.csv",
    index=False
)


PRELIMINARY CASH VS MORTGAGE COMPARISON
           neighbourhood          property_category  cap_rate  cash_annual_cash_flow  mortgage_annual_cash_flow  mortgage_cash_on_cash
                Downtown Entire Apartments / Condos    0.0208               13389.60                -20440.6257                -0.1272
               Killarney              Entire Houses    0.0082               16266.25                -87986.6151                -0.1776
               Kitsilano              Entire Houses    0.0077               22038.10               -127646.1801                -0.1795
          Mount Pleasant Entire Apartments / Condos    0.0066                4805.85                -33443.8764                -0.1840
               Kitsilano Entire Apartments / Condos    0.0054                5033.70                -43916.1375                -0.1888
                 Marpole              Entire Houses    0.0049               11340.00               -109751.6889                -0.1907
              

# Fill missing acquisition-value matches

In [80]:
# Provisional geographic crosswalk.
# These are approximations for the experiment, NOT verified boundary-level matches.

geo_proxy = {
    "Arbutus Ridge": ["Arbutus / Mackenzie Heights"],
    "Downtown Eastside": ["Harbour", "Hastings East"],
    "Dunbar Southlands": ["Dunbar", "Southlands"],
    "Grandview-Woodland": ["Grandview"],
    "Hastings-Sunrise": ["Hastings East"],
    "Kensington-Cedar Cottage": ["Cedar Cottage"],
    "Renfrew-Collingwood": ["Renfrew", "Collingwood"],
    "Riley Park": ["Main / Fraser"],
    "South Cambie": ["Cambie"],
    "Sunset": ["South Vancouver"],
    "Victoria-Fraserview": ["Fraserview"],
    "West Point Grey": ["Point Grey"],
    "Strathcona": ["Harbour", "Hastings East"]
}

# Start with all 27 ROI groups
roi_complete = roi_screening.copy()

# Ensure assessment names exist on property records
property_clean["assessment_neighbourhood"] = (
    property_clean["NEIGHBOURHOOD_CODE"]
    .map(assessment_neighbourhoods)
)

# Use individual assessed values when combining areas.
# This avoids averaging neighbourhood medians.

for idx, row in roi_complete.iterrows():

    if pd.notna(row["assessed_price_proxy"]):
        roi_complete.loc[idx, "match_method"] = "Exact name"
        continue

    neighbourhood = row["neighbourhood"]

    if neighbourhood not in geo_proxy:
        roi_complete.loc[idx, "match_method"] = "Unmatched"
        continue

    areas = geo_proxy[neighbourhood]

    comparable_properties = property_clean.loc[
        property_clean["assessment_neighbourhood"].isin(areas)
        & property_clean["LEGAL_TYPE"].eq(
            row["assessment_legal_type"]
        )
    ]

    if comparable_properties.empty:
        roi_complete.loc[idx, "match_method"] = "Unmatched"
        continue

    roi_complete.loc[idx, "assessed_price_proxy"] = (
        comparable_properties["assessed_value"].median()
    )

    roi_complete.loc[idx, "assessment_property_count"] = (
        len(comparable_properties)
    )

    roi_complete.loc[idx, "match_method"] = "Approximate area proxy"

roi_complete["gross_yield_proxy"] = (
    roi_complete["median_annual_revenue"]
    / roi_complete["assessed_price_proxy"]
)

roi_complete["price_match_status"] = np.where(
    roi_complete["assessed_price_proxy"].notna(),
    "Matched",
    "Unmatched"
)

# Print coverage and a compact comparison
print("\nMATCH COVERAGE")
print(roi_complete["match_method"].value_counts(dropna=False))

print("\nGROUPS WITHOUT PRICES")
print(
    roi_complete.loc[
        roi_complete["assessed_price_proxy"].isna(),
        ["neighbourhood", "property_category"]
    ].to_string(index=False)
)

print("\nCOMPLETE GROSS YIELD SCREENING")
print(
    roi_complete[
        [
            "neighbourhood",
            "property_category",
            "listings",
            "median_annual_revenue",
            "assessed_price_proxy",
            "gross_yield_proxy",
            "match_method"
        ]
    ]
    .sort_values("gross_yield_proxy", ascending=False)
    .round(4)
    .to_string(index=False)
)

Path("output").mkdir(exist_ok=True)

roi_complete.to_csv(
    "output/roi_complete_geography.csv",
    index=False
)



MATCH COVERAGE
match_method
Approximate area proxy    15
Exact name                12
Name: count, dtype: int64

GROUPS WITHOUT PRICES
Empty DataFrame
Columns: [neighbourhood, property_category]
Index: []

COMPLETE GROSS YIELD SCREENING
           neighbourhood          property_category  listings  median_annual_revenue  assessed_price_proxy  gross_yield_proxy           match_method
                Downtown Entire Apartments / Condos      1275                19128.0              643000.0             0.0297             Exact name
       Downtown Eastside Entire Apartments / Condos       308                34263.0             1526500.0             0.0224 Approximate area proxy
              Riley Park Entire Apartments / Condos        22                15353.5              838000.0             0.0183 Approximate area proxy
        Hastings-Sunrise              Entire Houses       138                22909.0             1683400.0             0.0136 Approximate area proxy
     Renfrew-Coll


# Build $10M Investment Portfolios

In [81]:
TOTAL_BUDGET = 10_000_000
CASH_RESERVE_RATE = 0.10
MORTGAGE_RESERVE_RATE = 0.15
CLOSING_COST_RATE = 0.02
OPERATING_EXPENSE_RATE = 0.30

MORTGAGE_DOWN_PAYMENT = 0.25
MORTGAGE_INTEREST_RATE = 0.05
AMORTIZATION_YEARS = 25

MAX_PROPERTIES_PER_GROUP = 3

def get_available_capital(financing):
    reserve_rate = (
        CASH_RESERVE_RATE
        if financing == "Cash"
        else MORTGAGE_RESERVE_RATE
    )
    return TOTAL_BUDGET * (1 - reserve_rate)


# Prepare investment candidates
candidates = roi_complete.copy()

candidates = candidates.loc[
    candidates["assessed_price_proxy"].gt(0)
    & candidates["median_annual_revenue"].notna()
].copy()

#downpayment

def get_down_payment_rate(property_price):

    if property_price <= 500_000:
        return 0.05
    
    elif property_price <= 1_000_000:
        return 0.20

    elif property_price <= 2_000_000:
        return 0.25

    elif property_price <= 3_000_000:
        return 0.35

    else:
        return 0.50



def build_portfolio(financing):

    options = []

    for _, row in candidates.iterrows():

        price = float(row["assessed_price_proxy"])
        revenue = float(row["median_annual_revenue"])
        expenses = revenue * OPERATING_EXPENSE_RATE

        down_rate = (
            1.0 if financing == "Cash"
            else get_down_payment_rate(price)
        )

        financials = calculate_investment(
            property_price=price,
            annual_revenue=revenue,
            annual_operating_expenses=expenses,
            down_payment_rate=down_rate,
            annual_interest_rate=MORTGAGE_INTEREST_RATE,
            amortization_years=AMORTIZATION_YEARS
        )

        closing_cost = price * CLOSING_COST_RATE

        capital_required = (
            financials["down_payment"] + closing_cost
        )

        options.append({
            "neighbourhood": row["neighbourhood"],
            "property_category": row["property_category"],
            "match_method": row["match_method"],
            "purchase_price": price,
            "annual_revenue": revenue,
            "annual_operating_expenses": expenses,
            "annual_noi": financials["net_operating_income"],
            "annual_debt_service":
                financials["annual_mortgage_payment"],
            "annual_cash_flow": financials["annual_cash_flow"],
            "initial_cash_required": capital_required,
            "annual_cash_yield": (
                financials["annual_cash_flow"] /
                capital_required
            ),
            "down_payment_rate": down_rate,
            "down_payment_amount": financials["down_payment"],
            "loan_amount": financials["loan_amount"],
        })

    options = pd.DataFrame(options)

    # Rank candidates by annual cash return on capital.
    # This is a greedy heuristic, not a global optimum.
    options = options.sort_values(
        "annual_cash_yield",
        ascending=False
    )

    selected = []
    capital_left = get_available_capital(financing)

    for _, option in options.iterrows():

        unit_cost = option["initial_cash_required"]

        affordable = int(
            (capital_left + 1e-6) // unit_cost
        )

        quantity = min(
            affordable,
            MAX_PROPERTIES_PER_GROUP
        )

        if quantity <= 0:
            continue

        selected_row = option.to_dict()
        selected_row["quantity"] = quantity

        selected.append(selected_row)

        capital_left -= quantity * unit_cost

    if not selected:
        return pd.DataFrame(), options

    portfolio = pd.DataFrame(selected)

    # Calculate group totals
    for metric in [
        "purchase_price",
        "annual_revenue",
        "annual_operating_expenses",
        "annual_noi",
        "annual_debt_service",
        "annual_cash_flow",
        "initial_cash_required"
    ]:
        portfolio["total_" + metric] = (
            portfolio[metric] * portfolio["quantity"]
        )

    return portfolio, options


cash_portfolio, cash_options = build_portfolio("Cash")
mortgage_portfolio, mortgage_options = build_portfolio("Mortgage")

print(
    mortgage_portfolio[
        [
            "neighbourhood",
            "property_category",
            "quantity",
            "purchase_price",
            "down_payment_rate",
            "down_payment_amount",
            "loan_amount",
            "annual_cash_flow"
        ]
    ].to_string(index=False)
)


def summarize_portfolio(portfolio, name):

    if portfolio.empty:
        return {
            "Strategy": name,
            "Properties": 0,
            "Initial Cash Used": 0,
            "2027 Revenue": 0,
            "2027 NOI": 0,
            "2027 Debt Service": 0,
            "2027 Cash Flow": 0,
            "Cash-on-Cash Return": np.nan
        }

    capital = portfolio["total_initial_cash_required"].sum()
    cashflow = portfolio["total_annual_cash_flow"].sum()

    return {
        "Strategy": name,
        "Properties": int(portfolio["quantity"].sum()),
        "Initial Cash Used": capital,
        "2027 Revenue":
            portfolio["total_annual_revenue"].sum(),
        "2027 NOI":
            portfolio["total_annual_noi"].sum(),
        "2027 Debt Service":
            portfolio["total_annual_debt_service"].sum(),
        "2027 Cash Flow": cashflow,
        "Cash-on-Cash Return": cashflow / capital
    }


portfolio_summary = pd.DataFrame([
    summarize_portfolio(cash_portfolio, "All Cash"),
    summarize_portfolio(mortgage_portfolio, "Mortgage")
])

print("\n$10 MILLION PORTFOLIO COMPARISON")
print(portfolio_summary.round(3).to_string(index=False))

print("\nALL-CASH PROPERTY ALLOCATION")
print(
    cash_portfolio[
        [
            "neighbourhood",
            "property_category",
            "quantity",
            "purchase_price",
            "annual_revenue",
            "annual_cash_flow",
            "total_initial_cash_required"
        ]
    ].round(2).to_string(index=False)
)

print("\nMORTGAGE PROPERTY ALLOCATION")
print(
    mortgage_portfolio[
        [
            "neighbourhood",
            "property_category",
            "quantity",
            "purchase_price",
            "annual_revenue",
            "annual_cash_flow",
            "total_initial_cash_required"
        ]
    ].round(2).to_string(index=False)
)

Path("output").mkdir(exist_ok=True)

portfolio_summary.to_csv(
    "output/portfolio_comparison.csv",
    index=False
)

cash_portfolio.to_csv(
    "output/cash_portfolio.csv",
    index=False
)

mortgage_portfolio.to_csv(
    "output/mortgage_portfolio.csv",
    index=False
)


neighbourhood          property_category  quantity  purchase_price  down_payment_rate  down_payment_amount  loan_amount  annual_cash_flow
 South Cambie              Entire Houses         3       3029000.0                0.5            1514500.0    1514500.0     -82956.494144
   Kerrisdale              Entire Houses         2       3421000.0                0.5            1710500.0    1710500.0    -103969.951920
     Downtown Entire Apartments / Condos         1        643000.0                0.2             128600.0     514400.0     -22695.974082

$10 MILLION PORTFOLIO COMPARISON
Strategy  Properties  Initial Cash Used  2027 Revenue  2027 NOI  2027 Debt Service  2027 Cash Flow  Cash-on-Cash Return
All Cash           8          8348190.0      190880.0  133616.0               0.00       133616.00                0.016
Mortgage           6          8424540.0      164709.0  115296.3          594801.66      -479505.36               -0.057

ALL-CASH PROPERTY ALLOCATION
    neighbourhood       

# Portfolio Cash Flow Stress Test

In [82]:
scenarios = {
    "Pessimistic": 0.70,  # Revenue 30% below base
    "Base": 1.00,
    "Optimistic": 1.30   # Revenue 30% above base
}

stress_results = []

for name, portfolio in [
    ("All Cash", cash_portfolio),
    ("Mortgage", mortgage_portfolio)
]:

    initial_cash = (
        portfolio["total_initial_cash_required"].sum()
    )

    remaining_cash = TOTAL_BUDGET - initial_cash

    for scenario, multiplier in scenarios.items():

        annual_revenue = (
            portfolio["total_annual_revenue"].sum()
            * multiplier
        )

        # Assumption: operating expenses remain 30%
        # of revenue in each scenario.
        operating_expenses = (
            annual_revenue * OPERATING_EXPENSE_RATE
        )

        debt_service = (
            portfolio["total_annual_debt_service"].sum()
        )

        cash_flow = (
            annual_revenue
            - operating_expenses
            - debt_service
        )

        ending_cash = remaining_cash + cash_flow

        stress_results.append({
            "Strategy": name,
            "Scenario": scenario,
            "2027 Revenue": annual_revenue,
            "2027 Cash Flow": cash_flow,
            "How much money we have left at the end of the year": ending_cash,
            "Budget Feasible": ending_cash >= 0
        })

stress_test = pd.DataFrame(stress_results)

print("\n2027 PORTFOLIO STRESS TEST")
print(stress_test.round(2).to_string(index=False))

stress_test.to_csv(
    "output/portfolio_stress_test.csv",
    index=False
)



2027 PORTFOLIO STRESS TEST
Strategy    Scenario  2027 Revenue  2027 Cash Flow  How much money we have left at the end of the year  Budget Feasible
All Cash Pessimistic      133616.0        93531.20                                          1745341.20             True
All Cash        Base      190880.0       133616.00                                          1785426.00             True
All Cash  Optimistic      248144.0       173700.80                                          1825510.80             True
Mortgage Pessimistic      115296.3      -514094.25                                          1061365.75             True
Mortgage        Base      164709.0      -479505.36                                          1095954.64             True
Mortgage  Optimistic      214121.7      -444916.47                                          1130543.53             True
